As of LangChain v1, initialize_agent and AgentType are deprecated and have moved to the langchain-classic package. The new standard for building agents is create_agent — a single, unified, middleware-driven agent constructor built on LangGraph, replacing langgraph.prebuilt.create_react_agent as well as every AgentType variant.

Each section below reproduces the intent of one of the original 7 agent types using create_agent, plus a bonus section on middleware — the core new capability in v1 that replaces most of what the old AgentType variants used to hard-code.

Reference: https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent

Migration cheat-sheet: AgentType → LangChain v1
#	Legacy AgentType (pre-v1)	                   What it did	                                           v1 equivalent with create_agent
1	ZERO_SHOT_REACT_DESCRIPTION	            Single-shot ReAct loop, no memory	                create_agent(model, tools) — the default agent loop
2	CONVERSATIONAL_REACT_DESCRIPTION	    ReAct + ConversationBufferMemory	                create_agent(..., checkpointer=InMemorySaver()) thread_id
3	CHAT_CONVERSATIONAL_REACT_DESCRIPTION	  Chat-model ReAct + memory	                           Same as above — chat models are the default everywhere now
4	STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION	   Multi-argument tools via a structured schema	     Any @tool-decorated function passed straight to create_agent
5	OPENAI_FUNCTIONS	                  Function-calling instead of text-based ReAct parsing	     Native tool calling — create_agent always uses the model's function/tool-calling API when available
6	OPENAI_MULTI_FUNCTIONS	             Multiple function calls in a single step	                 Native parallel tool calling — no special agent type needed
7	SELF_ASK_WITH_SEARCH	            Explicit Follow up: ... / Intermediate answer: ... prompting pattern	A system_prompt that encodes the self-ask pattern, combined with a search tool

Other things that moved or were renamed:

langchain.chat_models.ChatOpenAI → langchain_openai.ChatOpenAI (or langchain.chat_models.init_chat_model)
langchain.chains.LLMChain → moved to langchain-classic; prefer a plain @tool function or an LCEL chain
langchain.memory.ConversationBufferMemory → replaced by LangGraph checkpointers (InMemorySaver, PostgresSaver, ...)
langchain.tools.TavilySearchResults (community, deprecated) → langchain_tavily.TavilySearch


In [1]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# Load environment variables
load_dotenv(dotenv_path=".env")

api_key = os.getenv("OPENAI_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

# Initialize model
model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=api_key,
    use_responses_api=True
)

# Create QA tool
@tool
def simple_qa(question: str) -> str:
    """Answer a question clearly and directly."""

    response = model.invoke(question)

    answer = ""

    for item in response.content:
        if isinstance(item, dict) and item.get("type") == "text":
            answer += item.get("text", "")

    return answer
    
# Create web search tool
web_search = TavilySearch(
    max_results=3
)

# Combine tools
tools = [simple_qa, web_search]

C:\Anaconda3\envs\langchain\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


1) Basic agent (replaces ZERO_SHOT_REACT_DESCRIPTION)
The zero-shot ReAct agent was the "simple + reliable default" in the legacy API — no memory, single-pass reasoning + tool use. In v1 this is simply create_agent with no checkpointer and no extra middleware.

In [2]:
#)1.Basic agent (replaces ZERO_SHOT_REACT_DESCRIPTION)
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when they help you answer accurately.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]}
)

final_message = result["messages"][-1]

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        print(item["text"])

LangChain is an open-source framework for building applications powered by large language models.  
It helps connect models with data, tools, and workflows.

**Creator:** Harrison Chase.


2) Agent with conversational memory (replaces CONVERSATIONAL_REACT_DESCRIPTION)
ConversationBufferMemory is gone. Short-term memory in v1 is handled by a checkpointer attached to the agent, addressed by a thread_id. Every invoke() call on the same thread automatically sees the full prior message history — no manual memory object to wire up.



In [3]:
# 2) Agent with conversational memory (replaces `CONVERSATIONAL_REACT_DESCRIPTION`)
checkpointer = InMemorySaver()

memory_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
)

thread_config = {"configurable": {"thread_id": "agent-types-demo-1"}}

result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]},
    thread_config,
)
final_message = result["messages"][-1]

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        print(item["text"])

LangChain is an open-source framework for building applications powered by large language models.  
It helps developers connect models to data, tools, and multi-step workflows.

It was created by **Harrison Chase**.


3) Multi-turn chat agent (replaces CHAT_CONVERSATIONAL_REACT_DESCRIPTION)
CHAT_CONVERSATIONAL_REACT_DESCRIPTION mainly existed to point the conversational agent at a chat model instead of a plain text-completion LLM. In v1 every model is a chat model, so that distinction disappears — the memory-backed agent above already gives us multi-turn chat. Here we simply continue the same thread to show a follow-up question that depends on prior context:

In [4]:
## 3) Multi-turn chat agent (replaces `CHAT_CONVERSATIONAL_REACT_DESCRIPTION`)
follow_up = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Now plan a 3-step study path for what I just asked about."}]},
    thread_config,  # same thread_id -> agent remembers the prior turn
)
final_message = follow_up["messages"][-1]

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        print(item["text"])

1. **Learn the basics:** Understand what LangChain does—connecting language models with data, tools, and workflows. Review its core concepts and terminology.
2. **Follow a hands-on tutorial:** Use the official LangChain docs to build a small app, such as a chatbot that answers questions from your own documents.
3. **Build and evaluate a project:** Add a tool or multi-step workflow, test how well it works, and note where LangChain helps or adds complexity.


4) Structured, multi-argument tools (replaces STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)
StructuredTool + a Pydantic args_schema used to require its own dedicated AgentType. In v1, every tool passed to create_agent supports multi-argument, typed, structured calling out of the box — no special agent type required. The @tool decorator infers the schema straight from your function's type hints and docstring.

In [5]:
## 4) Structured, multi-argument tools (replaces `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION`)
@tool
def title_maker(topic: str, tone: str = "concise") -> str:
    """Generate a title given a topic and an optional tone (e.g. 'concise', 'playful', 'formal')."""
    return f"{tone.title()} Title: {topic} in Practice"

structured_agent = create_agent(
    model=model,
    tools=[title_maker],
)

result = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "Make a friendly title about LangGraph tutorials."}]}
)
final_message = result["messages"][-1]

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        print(item["text"])

LangGraph Tutorials: Learn by Building!


5) Native function / tool calling (replaces OPENAI_FUNCTIONS)
OPENAI_FUNCTIONS opted an agent into OpenAI's function-calling schema instead of brittle text-based ReAct parsing. create_agent always uses the model's native tool-calling API when the model supports it — there's nothing extra to configure. Streaming the run lets us inspect the intermediate AIMessage.tool_calls to see the function-calling schema in action:

isinstance() is a built-in Python function used to check whether an object belongs to a specific data type or class. It returns True or False.

In [7]:
## 5) Native function / tool calling (replaces `OPENAI_FUNCTIONS`)
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Search the web for LangGraph docs and give me 3 bullets."}]},
    stream_mode="values",
):
    last = step["messages"][-1]

    # Print only the final AI response
    if last.type == "ai" and not last.tool_calls and last.content:

        if isinstance(last.content, str):
            print(last.content)

        else:
            for item in last.content:
                if isinstance(item, dict) and item.get("type") == "text":
                    print(item["text"])

- **LangGraph** is a framework for building stateful, long-running agents and workflows, with explicit control over how they execute. [Overview](https://docs.langchain.com/oss/python/langgraph/overview)
- Its **Graph API** lets you define state, nodes, and edges to create sequences, branches, and loops. [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- The docs include guides for **memory, time travel, subgraphs, and workflow/agent patterns**. [Documentation index](https://docs.langchain.com/oss/python/langgraph/overview)


6) Parallel tool calls (replaces OPENAI_MULTI_FUNCTIONS)
OPENAI_MULTI_FUNCTIONS let the agent call more than one tool within a single step. This is native behavior in v1 — when a prompt needs several tools at once, the model can emit multiple tool_calls on a single AIMessage, and create_agent executes them together, no special agent type needed.

In [9]:
## 6) Parallel tool calls (replaces `OPENAI_MULTI_FUNCTIONS`)
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use the simple_qa tool to explain what LangChain is in one sentence, "
                    "AND separately use web_search to find the latest LangChain release version."
                ),
            }
        ]
    }
)

for msg in result["messages"]:
    if getattr(msg, "tool_calls", None):
        print(f"{len(msg.tool_calls)} tool call(s) in this step:", [tc["name"] for tc in msg.tool_calls])

final_message = result["messages"][-1]

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        print("\nFinal answer:\n",item["text"])

2 tool call(s) in this step: ['simple_qa', 'tavily_search']

Final answer:
 **What LangChain is:** LangChain is a framework for building applications powered by large language models by connecting them to data, tools, and workflows.

**Latest release:** GitHub’s [LangChain releases page](https://github.com/langchain-ai/langchain/releases) lists **1.5.2**.


7) Self-ask with search (replaces SELF_ASK_WITH_SEARCH)
The legacy self-ask agent forced a strict Follow up: ... / Intermediate answer: ... text format and required a search tool literally named "Intermediate Answer". In v1 there's no dedicated agent type for this — instead we encode the same reasoning pattern in the system_prompt and let the model decide when to break a question into sub-questions before answering.

In [10]:
## 7) Self-ask with search (replaces `SELF_ASK_WITH_SEARCH`)
self_ask_prompt = """You are a careful research assistant that uses the self-ask technique:
1. Decide if the question needs to be broken into simpler follow-up questions.
2. For each follow-up question, use the web_search tool to find the answer.
3. Once you have every intermediate answer you need, combine them into one final answer.
Always ground factual claims in your search results."""

self_ask_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=self_ask_prompt,
)

result = self_ask_agent.invoke(
    {"messages": [{"role": "user", "content": "When was the first LangChain release and who founded it?"}]}
)
final_message = result["messages"][-1]

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        print(item["text"])

The first LangChain Python package was released on **October 24, 2022**, as a side project by **Harrison Chase**. The company was founded in **February 2023** by Chase and co-founder **Ankush Gola**. ([LangChain](https://www.langchain.com/blog/langchain-second-birthday), [LangChain about page](https://www.langchain.com/about))


Bonus: middleware — the real replacement for most AgentType behavior
In the legacy API, every new "flavor" of agent behavior (parsing-error recovery, memory summarization, human approval, etc.) needed its own AgentType. In v1, that customization surface is now middleware: composable hooks that wrap the model call, wrap each tool call, or run before/after an agent step.

A few prebuilt/custom middlewares that map to things the old notebook was doing manually or couldn't do at all:

handle_parsing_errors=True (used on the old self-ask agent) → a @wrap_tool_call middleware that catches tool exceptions and returns a friendly ToolMessage
Conversation memory getting too long → SummarizationMiddleware
Requiring a human to approve a sensitive tool call → HumanInTheLoopMiddleware
See the middleware guide for the full list of hooks (before_agent, before_model, wrap_model_call, wrap_tool_call, after_model, after_agent).



In [11]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

## Bonus: middleware — the real replacement for most `AgentType` behavior
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Equivalent to the old `handle_parsing_errors=True` flag."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: please rephrase your request. ({e})",
            tool_call_id=request.tool_call["id"],
        )


robust_agent = create_agent(
    model=model,
    tools=tools,
    middleware=[handle_tool_errors],
)

result = robust_agent.invoke(
    {"messages": [{"role": "user", "content": "Answer briefly from simple_qa; also check the web for any LangChain updates."}]}
)
final_message = result["messages"][-1]

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        print(item["text"])

LangChain is an open-source framework for building LLM applications that connect models to tools and data.

**Web update:** LangChain and LangGraph reached v1.0 in October 2025. LangChain introduced `create_agent`, requires Python 3.10+, and moved some older functionality to `langchain-classic`. [Official announcement](https://www.langchain.com/blog/langchain-langgraph-1dot0)


Reference diagrams (carried over from the original notebook)
The image cells below (agenttypes1.png, agenttype2.png, pro1.png, pro2.png, pro3.png) illustrated the legacy AgentType architectures in the original notebook. The image files themselves were not part of the uploaded notebook, so they won't render unless you place the original PNGs in the same folder as this notebook. Conceptually, the v1 replacement for all of these diagrams is the single create_agent + middleware graph loop described at the top of this notebook — one graph, customized with middleware, instead of seven separate agent classes.

Summary
initialize_agent + AgentType → deprecated, moved to langchain-classic
create_agent is now the single entry point for building agents in LangChain v1
Behavior differences between the 7 legacy agent types are now expressed via prompting choices, tool schemas, and middleware — not via separate agent classes
Memory is handled by checkpointers (InMemorySaver, PostgresSaver, ...) + thread_id, not ConversationBufferMemory
Function calling and parallel tool calls are the default behavior, not a special mode
For more detail, see the official release notes: https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent